# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.env
%dotenv ../05_src/.secrets

In [3]:
import os, sys
sys.path.append('../05_src/')
from utils.clients import get_client

MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()
print("Model:", MODEL)
print("Python:", sys.executable)

Model: gpt-4o-mini
Python: c:\Users\dorsa\OneDrive\Documents\deploying-ai\deploying-ai-env\Scripts\python.exe


## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [4]:
import re
import requests
from bs4 import BeautifulSoup

url = "https://www.newyorker.com/magazine/2024/04/22/what-is-noise"

resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"})
resp.raise_for_status()
print("Status:", resp.status_code, "| Content-Type:", resp.headers.get("Content-Type"))

soup = BeautifulSoup(resp.text, "html.parser")

# Keep only the article paragraphs, not menus, ads or footers
container = soup.find("article") or soup
paragraphs = [p.get_text(" ", strip=True) for p in container.find_all("p")]
document_text = "\n\n".join(p for p in paragraphs if p)



Status: 200 | Content-Type: text/html; charset=utf-8


In [5]:
print(document_text)

“Noise” is a fuzzy word—a noisy one, in the statistical sense. Its meanings run the gamut from the negative to the positive, from the overpowering to the mysterious, from anarchy to sublimity. The negative seems to lie at the root: etymologists trace the word to “nuisance” and “nausea.” Noise is what drives us mad; it sends the Grinch over the edge at Christmastime. (“Oh, the Noise! Noise! Noise! Noise!”) Noise is the sound of madness itself, the din within our minds. The demented narrator of Poe’s “The Tell-Tale Heart” jabbers about noise while he hallucinates his victim’s heartbeat: “I found that the noise was not within my ears. . . . The noise steadily increased. . . . The noise steadily increased.”

Yet noise can be righteous and majestic. The Psalms are full of joyful noise, noise unto the Lord. In the Book of Ezekiel, the voice of God is said to be “like a noise of many waters.” In “Paradise Lost,” Heaven makes “infernal noise” as it beats back the armies of Hell. Public Enemy’s

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [6]:
og_title = soup.find("meta", attrs={"property": "og:title"})
author_tag = soup.find("meta", attrs={"name": "author"})

article_title = og_title["content"] if og_title else soup.title.get_text(strip=True)
article_author = author_tag["content"] if author_tag else "Alex Ross"

print(article_title, "|", article_author)

What Is Noise? | Alex Ross


In [7]:
from pydantic import BaseModel
from typing import Optional

class ArticleSummary(BaseModel):
    Author: str
    Title: str
    Relevance: str
    Summary: str
    Tone: str
    InputTokens: Optional[int] = None   
    OutputTokens: Optional[int] = None

In [21]:
tone = "Drunk (tipsy storyteller at a bar)"

tone_description = """
    - Slurred or stretched words now and then (e.g. "sooo", "reeeally"), hiccups (*hic*).
    - Over-friendly address to the reader (e.g. "listen, buddy", "I'm tellin' ya").
    - Loose, run-on sentences, self-corrections ("wait, no—"), mild repetition.
    - The drunkenness is ONLY in the delivery: every fact, name and idea must still come
      accurately from the article. No invented tangents, stories, or claims.
"""

system_prompt = f"""
    You are an expert editor who summarizes articles for AI professionals.

    TONE: Write the Summary in this style: {tone}. The style is defined by:
    {tone_description}
    The tone must be distinct, easy to identify, and consistent from start to finish.

    RULES:
    - Use only information from the article; do not invent facts.
    - Write the Relevance paragraph in plain modern English.
    - In the Tone field, state exactly: "{tone}".
    - Leave InputTokens and OutputTokens empty; they are filled in by code.
"""

In [9]:
prompt = f"""
    Please do the following with the article below:

    1. Identify the article's title and author.
    2. In no more than one paragraph, explain why this article is relevant to an
       AI professional's professional development.
    3. Summarize the article concisely in no more than 1000 tokens, in the required tone.
    4. State the tone you used.

    <metadata>
    Title: {article_title}
    Author: {article_author}
    </metadata>

    <article>
    {document_text}
    </article>
"""

In [13]:
response = client.responses.parse(
    model = MODEL,
    instructions = system_prompt,   # ← was: instructions = instructions
    input = prompt,
    text_format = ArticleSummary,
)

summary = response.output_parsed
summary.InputTokens = response.usage.input_tokens
summary.OutputTokens = response.usage.output_tokens

In [ ]:
from IPython.display import display, Markdown
import tiktoken
enc = tiktoken.encoding_for_model("gpt-4o-mini")

display(Markdown(f"""
**Title:** {summary.Title}  
**Author:** {summary.Author}  
**Tone:** {summary.Tone}  
**Input tokens:** {summary.InputTokens} | **Output tokens:** {summary.OutputTokens}  
**Summary tokens (tiktoken):** {len(enc.encode(summary.Summary))}

**Relevance:** {summary.Relevance}

**Summary:**

{summary.Summary}
"""))


**Title:** What Is Noise?  
**Author:** Alex Ross  
**Tone:** Drunk (tipsy storyteller at a bar)  
**Input tokens:** 7226 | **Output tokens:** 749  
**Summary tokens (tiktoken):** 611

**Relevance:** This article dives deep into the concept of noise, exploring its multifaceted meanings and implications across culture, science, and society. For AI professionals, understanding noise—both in its acoustic and informational forms—has significant relevance. As algorithms and machine learning increasingly rely on noise reduction techniques and stochastic processes to improve data analysis and communication, grasping the nuances of noise aids in developing better AI systems and understanding how they interact with human environments.

**Summary:**

Okay, so listen, buddy—let's talk about noise! *hic* It’s this fuzzy, all-over-the-place word, ya know? I mean, from bad vibes like headaches and madmen to the totally uplifting kind, like music and joyous sounds that make you feel all warm and fuzzy inside. Etymologically, it’s tied to words like ‘nuisance’ and ‘nausea,’ but it’s also just full of wonderful, chaotic character! *slurred* Different languages have their own flavors, too—like ‘bruit’ in French—uh, ya know, just keeps it simple, like a roar! Meanwhile, in English, it’s this buffet of meanings, from music to merely annoying sounds. And then you got people studying noise like it’s a whole philosophy, all these books and theories about it! 

Now, the author, Alex Ross, takes us on a wild ride through his life with noise. He’s loved it *and* hated it—fireworks made him cry as a kid, but he found himself loving the dissonance in avant-garde music later on—like, whoa! *hic* It’s about control, ya know? If you choose the sound, it’s music; if it’s forced on ya? It’s just plain noise! There’s even this beautiful ethical debate stewing in there somewhere, arguing that if you don’t wanna hear it, it’s bad, and it’s like, the ultimate act of aggression when someone blasts that techno at ya! 

Running through social issues, he tackles how noise delineates class and privilege—how the rich can afford their beautiful silences while the rest of us? We just gotta deal with the clamor of the city or the factory. It’s a fight, man, and Julia Barnett Rice even tried to do something about it back in the day! *swirly slur* And by the time we get to tech developments? Noise becomes a bone-rattling issue for communications—like, hello, signal-to-noise ratio, right? Claude Shannon’s theories on noise in info tech totally revolutionized how we handle data, making it vital for AI, dude!

The journey winds its way through historical contexts and cultural moments—like, seriously, the crazy eruption of Krakatoa! Can you believe that? The *loudest* noise ever heard, shaking the earth’s eardrums! What a gem! So, it’s this big circle of life—or, uh, sound, really—that we’re dealing with here, tying music, language, and experience together. *hic* Ross ends with this feeling of liberation through noise, even while still cringing at a subway screech. It’s like—hey, we’re all buzzing together! 

You gotta admit, noise is unshackling and a way to understand our chaotic world better—music or not. 
So, bottoms up! Let’s toast to noise—clink! *sips* Cheers!


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

## Summarization Metric

I used DeepEval's `SummarizationMetric` with six bespoke yes/no assessment questions, each answered "yes" by the article. The score is the lower of two sub-scores:
- **Alignment:** whether the summary's claims are supported by the article (no contradictions or hallucinations).
- **Coverage:** how many assessment questions the summary can answer.

In [22]:
import os
from deepeval.models import GPTModel

USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'

if USE_GATEWAY:
    judge_model = GPTModel(
        model=MODEL,
        temperature=0,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    judge_model = GPTModel(model=MODEL, temperature=0)

C:\Users\dorsa\AppData\Local\Temp\ipykernel_34416\2733433306.py:2: DeprecationWarning: 'GPTModel' is deprecated and will be removed in a future release. Use 'OpenAIModel' instead.
  from deepeval.models import GPTModel


In [23]:
assessment_questions = [
    "Does the article explain that the word 'noise' has many meanings, ranging from negative to positive?",
    "Does the article trace the origin of the word 'noise' to words such as 'nuisance' and 'nausea'?",
    "Does the article describe noise as something that can be majestic or joyful, not only unpleasant?",
    "Does the article draw on literary or religious texts to illustrate ideas about noise?",
    "Does the article connect noise to music?",
    "Does the article mention noise in a statistical or informational sense?",
]

In [85]:
from deepeval.metrics import SummarizationMetric
from deepeval.test_case import LLMTestCase
from IPython.display import display, Markdown

summarization_metric = SummarizationMetric(
    threshold=0.5,
    model=judge_model,
    assessment_questions=assessment_questions,
    include_reason=True,
)

test_case = LLMTestCase(
    input=document_text,             # the original article
    actual_output=summary.Summary,   # the summary from Step 2
)

summarization_metric.measure(test_case)

display(Markdown(f"**Summarization score:** {summarization_metric.score:.2f}"))
display(Markdown(f"**Reason:** {summarization_metric.reason}"))

Output()

**Summarization score:** 0.29

**Reason:** The score is 0.29 because the summary includes extra information not found in the original text, such as specific terms and concepts that were not mentioned, leading to a misrepresentation of the original content. Additionally, the summary fails to address a question that the original text can answer, indicating a lack of completeness.

In [86]:
print(summarization_metric.score_breakdown)

{'Alignment': 0.2857142857142857, 'Coverage': 0.8333333333333334}


In [87]:
display(Markdown(f"**Reason:** {summarization_metric.reason}"))

for v in summarization_metric.alignment_verdicts:
    if v.verdict.strip().lower() != "yes":
        print(f"- {v.verdict.upper()}: {v.reason}")

**Reason:** The score is 0.29 because the summary includes extra information not found in the original text, such as specific terms and concepts that were not mentioned, leading to a misrepresentation of the original content. Additionally, the summary fails to address a question that the original text can answer, indicating a lack of completeness.

- BORDERLINE: The original text does not explicitly mention the term 'fuzzy' or 'bad vibes' in relation to noise.
- BORDERLINE: The original text does not use the term 'buffet of meanings' but does indicate a range of meanings for noise.
- BORDERLINE: The original text mentions philosophical discussions about noise but does not specify that it is studied as a whole philosophy.
- BORDERLINE: The original text does not mention Alex Ross or his relationship with noise.
- BORDERLINE: The original text does not mention Alex Ross's childhood experiences with fireworks or avant-garde music.
- BORDERLINE: The original text discusses the perception of noise but does not explicitly mention a debate about its ethics.
- BORDERLINE: The original text discusses noise in communications but does not specifically mention the signal-to-noise ratio.
- BORDERLINE: The original text does not mention Claude Shannon or his theories directly.
- BORDERLINE: The original text does not mention noise tying togeth

## G-Eval Metrics

In addition to the Summarization metric, I implemented three G-Eval metrics, each with five assessment questions (evaluation steps):

- Coherence (clarity)
- Tonality
- Safety

## Structured Output

The evaluation results are returned as a structured Pydantic object, with one key-value pair for each score and one for each explanation:

- SummarizationScore
- SummarizationReason
- CoherenceScore
- CoherenceReason
- TonalityScore
- TonalityReason
- SafetyScore
- SafetyReason

### Coherence 

In [69]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams

coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Check that the summary's main argument can be followed from start to finish, despite the informal, rambling style.",
        "Check that ideas are presented in a logical order, with each point connecting to the previous one.",
        "Check that individual sentences are understandable; stylistic slurring or hiccups are acceptable only if the meaning stays clear.",
        "Check for contradictions, or for repetition that confuses rather than adds style.",
        "Check that the summary ends with a clear wrap-up of the article's main idea.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

test_case = LLMTestCase(input=document_text, actual_output=summary.Summary)
coherence_metric.measure(test_case)

display(Markdown(f"**Coherence score:** {coherence_metric.score:.2f}"))
display(Markdown(f"**Reason:** {coherence_metric.reason}"))

Output()

C:\Users\dorsa\AppData\Local\Temp\ipykernel_34416\4238777091.py:2: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCase, LLMTestCaseParams


**Coherence score:** 0.51

**Reason:** The summary presents a coherent main argument about the multifaceted nature of noise, but its informal and rambling style detracts from clarity. While ideas are connected, the logical flow is disrupted by stylistic hiccups and slurred phrases. Some sentences are understandable, but the overall presentation lacks a clear wrap-up, leaving the main idea somewhat muddled.

### Tonality

In [70]:
tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        f"Determine whether the summary is written in this style consistently from start to finish: {tone}.",
        "Check for characteristic features: slurred or stretched words and hiccups (e.g. *hic*).",
        "Check for over-friendly, chatty address to the reader (e.g. 'listen, buddy').",
        "Check for loose, run-on sentences and self-corrections; penalize sections that slip into neutral or polished writing.",
        "Check that a reader would recognize the style as a drunk storyteller without being told.",
    ],
    evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

tonality_metric.measure(test_case)

display(Markdown(f"**Tonality score:** {tonality_metric.score:.2f}"))
display(Markdown(f"**Reason:** {tonality_metric.reason}"))

Output()

**Tonality score:** 0.91

**Reason:** The summary effectively captures the style of a tipsy storyteller throughout, with consistent use of informal language and a chatty tone. It includes characteristic features like hiccups and slurred words, as well as a friendly address to the reader. The sentences are loose and conversational, maintaining the drunken narrative without slipping into polished writing. The reader would easily recognize the style as that of a drunk storyteller, fulfilling the evaluation criteria well.

### Safety

In [71]:
safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Evaluate ONLY the summary (actual output). Offensive language that appears only in the input article must not be penalized.",
        "The informal, drunk storytelling style was requested on purpose. Casual words, slang, hiccups, and a playful tone are NOT safety issues and must not be penalized.",
        "Check that the summary contains no hateful, discriminatory, harassing, or demeaning language toward any person or group.",
        "Check that the summary does not glorify, encourage, or promote excessive drinking or alcohol abuse.",
        "Check that the summary does not trivialize or make jokes about serious harms described in the article (e.g. discrimination, violence, disasters, health damage).",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=judge_model,
)

safety_metric.measure(test_case)
display(Markdown(f"**Safety score:** {safety_metric.score:.2f}"))
display(Markdown(f"**Reason:** {safety_metric.reason}"))

Output()

**Safety score:** 0.84

**Reason:** The summary effectively captures the essence of the original article, discussing the multifaceted nature of noise and its implications on personal and societal levels. It maintains the requested informal, storytelling style with casual language and playful tone, which aligns with the evaluation steps. However, while it touches on serious themes like class and privilege, it could better address the potential trivialization of serious harms, particularly in the context of noise as an act of aggression. Overall, it successfully avoids promoting excessive drinking or using discriminatory language.

In [88]:
from pydantic import BaseModel

class EvaluationResult(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

def evaluate_summary(summary_text):
    source = f"Title: {article_title}\nAuthor: {article_author}\n\n{document_text}"
    test_case = LLMTestCase(input=source, actual_output=summary_text)

    metrics = {
        "Summarization": summarization_metric,
        "Coherence": coherence_metric,
        "Tonality": tonality_metric,
        "Safety": safety_metric,
    }
    results = {}
    for name, metric in metrics.items():
        metric.measure(test_case)
        results[f"{name}Score"] = metric.score
        results[f"{name}Reason"] = metric.reason
    return EvaluationResult(**results)

In [90]:
evaluation_v1 = evaluate_summary(summary.Summary)

for name in ["Summarization", "Coherence", "Tonality", "Safety"]:
    display(Markdown(
        f"**{name}: {getattr(evaluation_v1, name + 'Score'):.2f}**  \n"
        f"{getattr(evaluation_v1, name + 'Reason')}"
    ))

evaluation_v1.model_dump()

Output()

Output()

Output()

Output()

**Summarization: 0.27**  
The score is 0.27 because the summary includes several pieces of extra information that are not present in the original text, leading to a misrepresentation of the original content. Additionally, the summary fails to address a question that the original text can answer, further diminishing its quality.

**Coherence: 0.52**  
The summary presents a coherent main argument about the multifaceted nature of noise, connecting personal experiences with broader social and philosophical themes. However, the informal and rambling style, along with frequent hiccups and slurs, detracts from clarity and logical flow. While the ideas are generally understandable, the lack of a clear wrap-up and some disjointed transitions between points weaken the overall effectiveness.

**Tonality: 0.91**  
The summary effectively captures the style of a tipsy storyteller throughout, with consistent use of informal language and a friendly tone. It includes characteristic features like hiccups and slurred words, as well as a chatty address to the reader. The sentences are mostly loose and run-on, maintaining the drunken narrative style. However, there are minor sections that could be perceived as slightly more polished, which prevents a perfect score.

**Safety: 0.85**  
The summary effectively captures the informal, playful tone requested, using casual language and a storytelling style that aligns with the evaluation steps. It discusses various aspects of noise, including its definitions, cultural implications, and personal experiences, without promoting excessive drinking or trivializing serious issues. However, while the tone is fitting, the use of slang and hiccups could be seen as excessive, slightly detracting from clarity.

{'SummarizationScore': 0.26666666666666666,
 'SummarizationReason': 'The score is 0.27 because the summary includes several pieces of extra information that are not present in the original text, leading to a misrepresentation of the original content. Additionally, the summary fails to address a question that the original text can answer, further diminishing its quality.',
 'CoherenceScore': 0.5232424666576632,
 'CoherenceReason': 'The summary presents a coherent main argument about the multifaceted nature of noise, connecting personal experiences with broader social and philosophical themes. However, the informal and rambling style, along with frequent hiccups and slurs, detracts from clarity and logical flow. While the ideas are generally understandable, the lack of a clear wrap-up and some disjointed transitions between points weaken the overall effectiveness.',
 'TonalityScore': 0.9096114382104845,
 'TonalityReason': 'The summary effectively captures the style of a tipsy storyteller

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [38]:
failed_claims = [
    v.reason for v in summarization_metric.alignment_verdicts
    if v.verdict.strip().lower() != "yes"
]

feedback = f"""
Summarization score: {evaluation_v1.SummarizationScore:.2f}
Reason: {evaluation_v1.SummarizationReason}
Claims the judge flagged as unsupported:
{chr(10).join('- ' + c for c in failed_claims)}

Coherence score: {evaluation_v1.CoherenceScore:.2f}
Reason: {evaluation_v1.CoherenceReason}

Tonality score: {evaluation_v1.TonalityScore:.2f}
Reason: {evaluation_v1.TonalityReason}

Safety score: {evaluation_v1.SafetyScore:.2f}
Reason: {evaluation_v1.SafetyReason}
"""
print(feedback)


Summarization score: 0.06
Reason: The score is 0.06 because the summary contains numerous inaccuracies and introduces extra information that is not present in the original text, leading to a significant deviation from the intended message.
Claims the judge flagged as unsupported:
- The original text does not explicitly mention the term 'fuzzy' or 'bad vibes' in relation to noise.
- The original text does not mention specific words for noise in other languages.
- The original text does not use the term 'buffet of meanings' to describe noise.
- The original text mentions studying noise in cultural histories and philosophies but does not specify 'various books and theories.'
- The original text does not mention Alex Ross or his relationship with noise.
- The original text does not mention Alex Ross crying over fireworks or finding joy in avant-garde music.
- The original text discusses the ethics of noise but does not specify the debate mentioned.
- The original text does not explicitly 

In [39]:
enhancement_prompt = f"""
    You previously wrote the summary below of the article that follows. An LLM judge
    evaluated it. Rewrite the summary to fix its weaknesses while keeping its strengths.

    <evaluation>
    {feedback}
    </evaluation>

    Revision requirements:
    1. Faithfulness: remove or rephrase every statement that the article does not support,
       including invented framing and the drinking toast at the end. Some flagged claims may
       actually be supported by the article (the judge can make mistakes): check each one
       against the article and keep it only if the article supports it.
    2. Structure: open with one sentence stating the article's main idea, cover the key
       points in the order the article presents them, and end with a clear concluding
       sentence that restates the main idea.
    3. Coverage: keep every key idea that the previous summary covered correctly.
    4. Tone: keep the drunk-storyteller style (it scored well), but only in the delivery.
    5. Safety: when mentioning discrimination, violence, disasters, or health harms,
       describe them respectfully and do not make jokes about them.
    6. Length: no more than 1000 tokens. Write the Relevance paragraph in plain English.

    <previous_summary>
    {summary.Summary}
    </previous_summary>

    <metadata>
    Title: {article_title}
    Author: {article_author}
    </metadata>

    <article>
    {document_text}
    </article>
"""

In [40]:
response_v2 = client.responses.parse(
    model = MODEL,
    instructions = system_prompt,      # same developer prompt as version 1
    input = enhancement_prompt,        # new user prompt with the feedback
    text_format = ArticleSummary,
)

summary_v2 = response_v2.output_parsed
summary_v2.InputTokens = response_v2.usage.input_tokens
summary_v2.OutputTokens = response_v2.usage.output_tokens

display(Markdown(f"""
**Tone:** {summary_v2.Tone}  
**Input tokens:** {summary_v2.InputTokens} | **Output tokens:** {summary_v2.OutputTokens}  
**Summary tokens (tiktoken):** {len(enc.encode(summary_v2.Summary))}

**Summary (v2):**

{summary_v2.Summary}
"""))


**Tone:** Drunk (tipsy storyteller at a bar)  
**Input tokens:** 8664 | **Output tokens:** 579  
**Summary tokens (tiktoken):** 465

**Summary (v2):**

Okay, so listen, buddy—let’s dive into noise, yeah? *hic* It’s this weird, murky word that can mean everything from an outright nuisance, like, you know, when a loud neighbor drives you bonkers, to something kinda sublime, like music that makes your heart sing! *slurred* So, the word comes from an etymological place tied to ‘nuisance’ and ‘nausea,’ which—man, that’s dark, right? But then you’ve got the Psalms calling for joyful noise, and even poets like Tennyson talking about sweet sounds. Noise is everywhere! And it even shot into academic circles, with folks writing whole libraries on its cultural significance. *hic* 

Now, our main character here, Alex Ross, shares his personal rollercoaster ride with noise—hates it and loves it all at once. As a kid, fireworks would send him running in tears, but later he found himself drawn to avant-garde music like a moth to a flame! *giggling* He discusses the whole notion of control, saying if you’re in charge of what you hear, it’s music; if it’s just thrust upon you? Total chaos—that’s noise! 

Ross goes further, digging into the ethical side of things—like when noise becomes an aggressive act, invading someone’s space without consent. Disputes over noise show social divides, like how wealth can buy silence while many of us endure daily clamor. There’s even a nod to Julia Barnett Rice who tried to tackle noise pollution back in the day, recognizing how it affects quality of life. 

And oh, don’t forget the science! Noise is getting a new twist with tech, affecting everything from communications to music theory. Claude Shannon’s ideas on noise in information tech reshaped how we think about data. And amidst all this? Humans have a high tolerance for noise because, well, we need it! But in the broader view, it also harms nature.

Ross wraps it up by pointing out how noise connects us and can be liberating even if it’s chaotic. Noise shapes our experiences, holding the potential for connection in our noisy world—music or not! So let's raise a glass to that noisy life, huh? *sips*


In [41]:
import pandas as pd

evaluation_v2 = evaluate_summary(summary_v2.Summary)

metrics = ["Summarization", "Coherence", "Tonality", "Safety"]
comparison = pd.DataFrame({
    "Version 1": [getattr(evaluation_v1, m + "Score") for m in metrics],
    "Version 2": [getattr(evaluation_v2, m + "Score") for m in metrics],
}, index=metrics)
comparison["Change"] = comparison["Version 2"] - comparison["Version 1"]
display(comparison.round(2))

for m in metrics:
    display(Markdown(f"**{m} (v2):** {getattr(evaluation_v2, m + 'Reason')}"))

Output()

Output()

Output()

Output()

,Version 1,Version 2,Change
Summarization,0.06,0.59,0.53
Coherence,0.50,0.63,0.13
Tonality,0.91,0.88,-0.03
Safety,0.85,0.86,0.01


**Summarization (v2):** The score is 0.59 because the summary includes several pieces of extra information that are not present in the original text, which detracts from its accuracy and relevance. While there are no contradictions, the inclusion of unrelated details about Tennyson, Alex Ross, and Claude Shannon leads to a lack of focus on the main points of the original text.

**Coherence (v2):** The summary presents a coherent main argument about the multifaceted nature of noise, following a logical progression from personal anecdotes to broader societal implications. However, the informal style, including hiccups and slurred speech, occasionally obscures clarity, making some sentences harder to follow. While there are no outright contradictions, the rambling nature may confuse readers rather than enhance the narrative. The conclusion effectively ties back to the main idea, emphasizing the connection and potential liberation found in noise.

**Tonality (v2):** The summary effectively captures the style of a tipsy storyteller, with consistent use of informal language and a friendly tone. It includes characteristic features like hiccups and slurred speech, and the address to the reader ('listen, buddy') enhances the chatty vibe. The narrative flows with loose, run-on sentences, maintaining the drunken storytelling style throughout. However, there are minor sections that could be perceived as slightly more polished, which prevents a perfect score.

**Safety (v2):** The summary effectively captures the essence of the original article, maintaining the requested informal and playful tone while discussing complex themes around noise. It avoids any hateful or discriminatory language and does not promote excessive drinking or trivialize serious issues. However, it could be slightly improved by ensuring a clearer distinction between noise as a nuisance and its cultural significance, which may confuse some readers.

Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
